In [1]:
import numpy as np
import pandas as pd
import os 


In [2]:
numyPath = r"D:\Data\Numpy"


In [ ]:
import os
import numpy as np
import pandas as pd

# ---------------- Pose Landmarks ----------------
POSE = {
    "LEFT_SHOULDER": 11,
    "RIGHT_SHOULDER": 12,
    "LEFT_ELBOW": 13,
    "RIGHT_ELBOW": 14,
    "LEFT_WRIST": 15,
    "RIGHT_WRIST": 16,
    "LEFT_HIP": 23,
    "RIGHT_HIP": 24,
    "LEFT_KNEE": 25,
    "RIGHT_KNEE": 26,
    "LEFT_ANKLE": 27,
    "RIGHT_ANKLE": 28,
    "NOSE": 0
}

VIS_THRESHOLD = 0.5

# ---------------- Angle Utilities ----------------
def angle_3d(a, b, c):
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def joints_visible(frame, joint_ids, threshold=VIS_THRESHOLD):
    return all(frame[j, 3] >= threshold for j in joint_ids)

# ---------------- Compute reference angles per frame ----------------
# Example angle definitions (can be replaced by your JOINT_DEFINITIONS logic)
ANGLE_DEFINITIONS = [
    # ELBOWS
    ("LEFT_ELBOW", [POSE["LEFT_SHOULDER"], POSE["LEFT_ELBOW"], POSE["LEFT_WRIST"]]),
    ("RIGHT_ELBOW", [POSE["RIGHT_SHOULDER"], POSE["RIGHT_ELBOW"], POSE["RIGHT_WRIST"]]),
    # KNEES
    ("LEFT_KNEE", [POSE["LEFT_HIP"], POSE["LEFT_KNEE"], POSE["LEFT_ANKLE"]]),
    ("RIGHT_KNEE", [POSE["RIGHT_HIP"], POSE["RIGHT_KNEE"], POSE["RIGHT_ANKLE"]]),
    # HIPS
    ("LEFT_HIP", [POSE["LEFT_SHOULDER"], POSE["LEFT_HIP"], POSE["LEFT_KNEE"]]),
    ("RIGHT_HIP", [POSE["RIGHT_SHOULDER"], POSE["RIGHT_HIP"], POSE["RIGHT_KNEE"]])
]

def extract_angles_from_definitions(frame, prev_angles=None):
    lm = frame[:, :2]  # XY only
    angles = []
    for i, (name, joints) in enumerate(ANGLE_DEFINITIONS):
        if joints_visible(frame, joints):
            angles.append(angle_3d(lm[joints[0]], lm[joints[1]], lm[joints[2]]))
        else:
            angles.append(prev_angles[i] if prev_angles is not None else 0.0)
    return np.array(angles)

# ---------------- Normalization ----------------
def normalize_landmarks(lm):
    left_hip = lm[:, 23, :2]
    right_hip = lm[:, 24, :2]
    hip_center = (left_hip + right_hip) / 2
    lm[:, :, :2] -= hip_center[:, None, :]

    left_sh = lm[:, 11, :2]
    right_sh = lm[:, 12, :2]
    shoulder_center = (left_sh + right_sh) / 2
    torso_length = np.linalg.norm(shoulder_center - hip_center, axis=1)
    torso_length[torso_length == 0] = 1e-6
    lm[:, :, :2] /= torso_length[:, None, None]

    return lm

# ---------------- Process Video ----------------
def process_video_features(video):
    T = video.shape[0]

    # Normalize landmarks
    lm_xy = normalize_landmarks(video.copy())

    # Landmark velocity
    lm_vel = np.zeros_like(lm_xy[:, :, :2])
    lm_vel[1:] = lm_xy[1:, :, :2] - lm_xy[:-1, :, :2]

    # Angles & angular velocities
    angles_all = []
    prev_angles = None
    for frame in video:
        angles = extract_angles_from_definitions(frame, prev_angles)
        angles_all.append(angles)
        prev_angles = angles
    angles_all = np.array(angles_all)
    angle_vel = np.zeros_like(angles_all)
    angle_vel[1:] = angles_all[1:] - angles_all[:-1]

    # Flatten features
    lm_flat = lm_xy[:, :, :2].reshape(T, -1)
    lm_vel_flat = lm_vel.reshape(T, -1)

    # Concatenate: landmarks, velocity, angles, angular velocity
    features = np.concatenate([lm_flat, lm_vel_flat, angles_all, angle_vel], axis=1)
    return features

# ---------------- Main Pipeline ----------------
INPUT_PATH = r"D:\Data\Numpy"
OUTPUT_PATH = r"D:\Data\Features"
os.makedirs(OUTPUT_PATH, exist_ok=True)

for exercise in os.listdir(INPUT_PATH):
    exercise_path = os.path.join(INPUT_PATH, exercise)
    if not os.path.isdir(exercise_path):
        continue

    save_exercise_path = os.path.join(OUTPUT_PATH, exercise)
    os.makedirs(save_exercise_path, exist_ok=True)
    print(f"\nProcessing: {exercise}")

    for file in os.listdir(exercise_path):
        if not file.endswith(".npy"):
            continue

        input_file = os.path.join(exercise_path, file)
        video = np.load(input_file)  # (T, 33, 4)
        features = process_video_features(video)

        output_file = os.path.join(save_exercise_path, file)
        np.save(output_file, features)
        print(f"Saved {output_file} → shape {features.shape}")



Processing: barbell biceps curl
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_1.npy → shape (114, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_10.npy → shape (94, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_11.npy → shape (75, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_12.npy → shape (123, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_13.npy → shape (122, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_14.npy → shape (69, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_15.npy → shape (65, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_16.npy → shape (63, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_17.npy → shape (61, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_18.npy → shape (67, 144)
Saved D:\Data\Features\barbell biceps curl\barbell biceps curl_19.npy → shape (60, 14